In [1]:
# ============================================================
# 01 - Data acquisition: DOGE/USDT historical candles from Binance
# ============================================================

import os
import pandas as pd
from binance.client import Client

# -----------------------------
# Configuration
# -----------------------------
SYMBOL = "DOGEUSDT"
INTERVAL = Client.KLINE_INTERVAL_5MINUTE

START_DATE = "1 Jan, 2017"
END_DATE = None  # None = hasta el momento actual

OUTPUT_DIR = "../data/raw"
OUTPUT_FILE = "DOGEUSDT_5m_binance_2017_2026.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Binance client
# Public historical data does not require API keys
# -----------------------------
client = Client()

# -----------------------------
# Download historical klines
# -----------------------------
klines = client.get_historical_klines(
    symbol=SYMBOL,
    interval=INTERVAL,
    start_str=START_DATE,
    end_str=END_DATE
)

# -----------------------------
# Build DataFrame
# -----------------------------
columns = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "number_of_trades",
    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
    "ignore"
]

df = pd.DataFrame(klines, columns=columns)

# -----------------------------
# Type conversions
# -----------------------------
df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

numeric_cols = [
    "open", "high", "low", "close", "volume",
    "quote_asset_volume", "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume"
]

df[numeric_cols] = df[numeric_cols].astype(float)
df["number_of_trades"] = df["number_of_trades"].astype(int)

df = df.drop(columns=["ignore"])

# -----------------------------
# Basic validation
# -----------------------------
print("Dataset downloaded successfully")
print(f"Shape: {df.shape}")
print(f"Date range: {df['open_time'].min()} -> {df['open_time'].max()}")
print(f"Duplicated timestamps: {df['open_time'].duplicated().sum()}")
print(f"Missing values: {df.isna().sum().sum()}")

display(df.head())
display(df.tail())

# -----------------------------
# Save raw dataset
# -----------------------------
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved to: {OUTPUT_PATH}")

Dataset downloaded successfully
Shape: (723409, 11)
Date range: 2019-07-05 12:00:00 -> 2026-05-23 11:10:00
Duplicated timestamps: 0
Missing values: 0


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004350,281690734.0,2019-07-05 12:04:59.999,1.213789e+06,1841,131946467.0,572788.485474
1,2019-07-05 12:05:00,0.004340,0.004340,0.004000,0.004022,151285521.0,2019-07-05 12:09:59.999,6.310450e+05,963,41885000.0,173783.101331
2,2019-07-05 12:10:00,0.004022,0.004069,0.003800,0.003870,182657540.0,2019-07-05 12:14:59.999,7.079442e+05,1285,58720314.0,228693.805868
3,2019-07-05 12:15:00,0.003861,0.003940,0.003859,0.003929,73986015.0,2019-07-05 12:19:59.999,2.885708e+05,535,43864831.0,171156.394506
4,2019-07-05 12:20:00,0.003929,0.003964,0.003870,0.003877,67030342.0,2019-07-05 12:24:59.999,2.627509e+05,399,27619614.0,108776.180726


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
723404,2026-05-23 10:50:00,0.09957,0.09961,0.09951,0.09952,2019387.0,2026-05-23 10:54:59.999,201036.76574,672,1267731.0,126207.39518
723405,2026-05-23 10:55:00,0.09952,0.09954,0.09950,0.09952,284994.0,2026-05-23 10:59:59.999,28361.11493,286,211374.0,21034.40172
723406,2026-05-23 11:00:00,0.09952,0.09955,0.09946,0.09948,591135.0,2026-05-23 11:04:59.999,58808.86284,561,257920.0,25658.84237
723407,2026-05-23 11:05:00,0.09948,0.09962,0.09947,0.09954,1837323.0,2026-05-23 11:09:59.999,182909.11863,1442,1026953.0,102221.07452
723408,2026-05-23 11:10:00,0.09954,0.09961,0.09954,0.09956,286859.0,2026-05-23 11:14:59.999,28558.43106,459,182916.0,18210.52168



Saved to: ../data/raw\DOGEUSDT_5m_binance_2017_2026.csv


In [2]:
# ============================================================
# 02 - Raw dataset validation
# ============================================================

import pandas as pd

df = pd.read_csv("../data/raw/DOGEUSDT_5m_binance_2017_2026.csv", parse_dates=["open_time", "close_time"])

print("Shape:", df.shape)
print("Date range:", df["open_time"].min(), "->", df["open_time"].max())
print("Duplicated open_time:", df["open_time"].duplicated().sum())
print("Missing values total:", df.isna().sum().sum())

display(df.info())
display(df.describe().T)
display(df.head())
display(df.tail())

Shape: (723409, 11)
Date range: 2019-07-05 12:00:00 -> 2026-05-23 11:10:00
Duplicated open_time: 0
Missing values total: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 723409 entries, 0 to 723408
Data columns (total 11 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   open_time                     723409 non-null  datetime64[ns]
 1   open                          723409 non-null  float64       
 2   high                          723409 non-null  float64       
 3   low                           723409 non-null  float64       
 4   close                         723409 non-null  float64       
 5   volume                        723409 non-null  float64       
 6   close_time                    723409 non-null  datetime64[ns]
 7   quote_asset_volume            723409 non-null  float64       
 8   number_of_trades              723409 non-null  int64         
 9   taker_buy_base_asset_vol

None

,count,mean,min,25%,50%,75%,max,std
open_time,723409,2022-12-14 04:37:11.707238144,2019-07-05 12:00:00,2021-03-25 23:10:00,2022-12-14 11:50:00,2024-09-02 12:10:00,2026-05-23 11:10:00,NaN
open,723409.0,0.115579,0.001219,0.05777,0.09129,0.16769,0.73623,0.100818
high,723409.0,0.11586,0.001238,0.057894,0.09141,0.16802,0.73995,0.101173
low,723409.0,0.115291,0.001135,0.057627,0.09115,0.16734,0.72743,0.100452
close,723409.0,0.115579,0.001218,0.05777,0.09129,0.16769,0.73623,0.100818
volume,723409.0,6057161.290995,0.0,785871.0,1987145.0,4898832.3,2174740295.0,22375722.713418
close_time,723409,2022-12-14 04:42:11.704691200,2019-07-05 12:04:59.999000,2021-03-25 23:14:59.999000064,2022-12-14 11:54:59.999000064,2024-09-02 12:14:59.999000064,2026-05-23 11:14:59.999000,NaN
quote_asset_volume,723409.0,960713.407364,0.0,36616.92197,191004.4613,632838.510296,378267072.445506,4014426.803393
number_of_trades,723409.0,2142.021385,0.0,125.0,592.0,2068.0,385041.0,5341.924789
taker_buy_base_asset_volume,723409.0,3003455.417293,0.0,345441.0,948678.0,2424087.0,1087817541.0,11265312.551116


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2019-07-05 12:00:00,0.004490,0.004600,0.003760,0.004350,281690734.0,2019-07-05 12:04:59.999,1.213789e+06,1841,131946467.0,572788.485474
1,2019-07-05 12:05:00,0.004340,0.004340,0.004000,0.004022,151285521.0,2019-07-05 12:09:59.999,6.310450e+05,963,41885000.0,173783.101331
2,2019-07-05 12:10:00,0.004022,0.004069,0.003800,0.003870,182657540.0,2019-07-05 12:14:59.999,7.079442e+05,1285,58720314.0,228693.805868
3,2019-07-05 12:15:00,0.003861,0.003940,0.003859,0.003929,73986015.0,2019-07-05 12:19:59.999,2.885708e+05,535,43864831.0,171156.394506
4,2019-07-05 12:20:00,0.003929,0.003964,0.003870,0.003877,67030342.0,2019-07-05 12:24:59.999,2.627509e+05,399,27619614.0,108776.180726


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
723404,2026-05-23 10:50:00,0.09957,0.09961,0.09951,0.09952,2019387.0,2026-05-23 10:54:59.999,201036.76574,672,1267731.0,126207.39518
723405,2026-05-23 10:55:00,0.09952,0.09954,0.09950,0.09952,284994.0,2026-05-23 10:59:59.999,28361.11493,286,211374.0,21034.40172
723406,2026-05-23 11:00:00,0.09952,0.09955,0.09946,0.09948,591135.0,2026-05-23 11:04:59.999,58808.86284,561,257920.0,25658.84237
723407,2026-05-23 11:05:00,0.09948,0.09962,0.09947,0.09954,1837323.0,2026-05-23 11:09:59.999,182909.11863,1442,1026953.0,102221.07452
723408,2026-05-23 11:10:00,0.09954,0.09961,0.09954,0.09956,286859.0,2026-05-23 11:14:59.999,28558.43106,459,182916.0,18210.52168


In [3]:
# ============================================================
# 03 - Check expected 5-minute frequency gaps
# ============================================================

df = df.sort_values("open_time").reset_index(drop=True)

df["time_diff"] = df["open_time"].diff()

gap_counts = df["time_diff"].value_counts().head(10)

print("Most common time differences:")
display(gap_counts)

gaps = df[df["time_diff"] != pd.Timedelta(minutes=5)].copy()

# The first row is always NaT, so we exclude it
gaps = gaps[gaps["time_diff"].notna()]

print(f"Number of detected gaps: {len(gaps)}")

display(gaps.head(20))

Most common time differences:


time_diff
0 days 00:05:00    723390
0 days 01:05:00         3
0 days 02:05:00         2
0 days 02:35:00         2
0 days 02:25:00         1
0 days 05:55:00         1
0 days 08:05:00         1
0 days 02:10:00         1
0 days 03:35:00         1
0 days 04:15:00         1
Name: count, dtype: int64

Number of detected gaps: 18


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume,time_diff
11688,2019-08-15 10:00:00,0.002684,0.002684,0.002619,0.002637,5177724.0,2019-08-15 10:04:59.999,1.372136e+04,72,1366733.0,3.588617e+03,0 days 08:05:00
37512,2019-11-13 04:20:00,0.002658,0.002658,0.002658,0.002658,0.0,2019-11-13 04:24:59.999,0.000000e+00,0,0.0,0.000000e+00,0 days 02:25:00
40940,2019-11-25 04:00:00,0.002189,0.002189,0.002165,0.002165,1347492.0,2019-11-25 04:04:59.999,2.929546e+03,39,23170.0,5.027890e+01,0 days 02:05:00
62804,2020-02-09 03:00:00,0.003246,0.003291,0.003233,0.003265,18708621.0,2020-02-09 03:04:59.999,6.137553e+04,132,4394114.0,1.435912e+04,0 days 01:05:00
65788,2020-02-19 17:30:00,0.002813,0.002824,0.002813,0.002819,3571441.0,2020-02-19 17:34:59.999,1.005028e+04,26,612208.0,1.725830e+03,0 days 05:55:00
69723,2020-03-04 11:30:00,0.002456,0.002456,0.002419,0.002419,3833351.0,2020-03-04 11:34:59.999,9.345212e+03,56,1946592.0,4.753604e+03,0 days 02:10:00
84585,2020-04-25 04:30:00,0.002080,0.002080,0.002080,0.002080,10035.0,2020-04-25 04:34:59.999,2.087681e+01,1,10035.0,2.087681e+01,0 days 02:35:00
102987,2020-06-28 05:30:00,0.002292,0.002292,0.002292,0.002292,4362.0,2020-06-28 05:34:59.999,9.999013e+00,1,4362.0,9.999013e+00,0 days 03:35:00
147633,2020-11-30 07:00:00,0.003519,0.003543,0.003515,0.003523,2165239.0,2020-11-30 07:04:59.999,7.628874e+03,111,693356.0,2.448435e+03,0 days 01:05:00
153763,2020-12-21 18:00:00,0.004567,0.005000,0.004567,0.004906,82218870.0,2020-12-21 18:04:59.999,3.987291e+05,1254,67667014.0,3.275852e+05,0 days 04:15:00
